In [1]:
import sys
sys.path.append(r'..\src')

In [2]:
import config
config = config.ProjectConfig()
print(config.random_state)

22


In [4]:
from data import load_data
features, properties = load_data('../data/features.xlsx', '../data/properties.xlsx')

In [5]:
# random forest
param_grid_rf = {'n_estimators': [100, 200, 250], 'max_depth': [5, 10, 15], 'min_samples_leaf':[5,7,10]}
# decision_tree
param_grid_dt = {'max_depth': [2, 5, 10], 'min_samples_leaf':[5,7,10], 'splitter':['best', 'random'],'min_samples_split':[10,20,30]}
# XGB
param_grid_xgb = {'n_estimators': [100, 200, 250], 'max_depth': [6, 10, 15], 'learning_rate':[0.01,0.05,0.1]}

In [9]:
from tuning import tune_models
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

dtreg = DecisionTreeRegressor()
rfreg = RandomForestRegressor()
xgbreg = XGBRegressor()

param_grid_list = [param_grid_dt, param_grid_xgb]
model_name_list = ['decision_tree', 'XGB']#'KNN']
model_instance_list = [dtreg, xgbreg]

# for estimator, param_grid, model_name in zip(model_instance_list, param_grid_list, model_name_list):
#     tune_models(features,properties,estimator,param_grid,config,model_name)

In [10]:
from evaluation import evaluate_models

In [12]:
from sklearn.base import clone
import json
from glob import glob
json_output_paths = glob(f"{config.output_dir}/*.json")
print(json_output_paths)

for estimator, output_path, model_name in zip(model_instance_list, json_output_paths, model_name_list):
#     print(estimator)
    with open(output_path, "r", encoding="utf-8") as file:
        best_params = json.load(file)
#     print(best_params['CO(conv)'])
#     model = clone(estimator).set_params(**best_params['CO(conv)'])
#     print(model)
    evaluate_models(features,properties,estimator,best_params,config,model_name)

['artifacts\\decision_tree_best_parameters.json', 'artifacts\\XGB_best_parameters.json']


In [14]:
from plotting import plot_performance
import pandas as pd
csv_output_paths = glob(f"{config.output_dir}/*.csv")
for csvs in csv_output_paths:
    results_df = pd.read_csv(csvs)
    plot_performance(results_df, config.plot_dir)